# Deep Model Diagnosis

This notebook performs a deep diagnosis of our best performing model (MiniLM + Tree-sitter -> Logistic Regression).
We investigate class-level performance, data scarcity, confusion pairs, confidence calibration, and problem-disjoint generalization.

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

sys.path.insert(0, '..')

DATA_DIR    = '../data/processed'
MODELS_DIR  = '../models'
RAW_DATA_DIR = '../data/raw/mcminer'

os.makedirs('../experiments/results/diagnosis', exist_ok=True)

## 1. Load Taxonomy and Splits

In [ ]:
# Load taxonomy
misc_bank = {}
bank_path = os.path.join(RAW_DATA_DIR, 'misconception_bank.json')
if os.path.exists(bank_path):
    with open(bank_path) as f:
        raw = json.load(f)
    for v in raw:
        misc_bank[int(v['misconception_id'])] = v['description']
        
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'validation.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
all_df   = pd.read_csv(os.path.join(DATA_DIR, 'mcminer_all.csv'))
pd_test_df = pd.read_csv(os.path.join(DATA_DIR, 'problem_disjoint_test.csv'))

## 2. Load Model and Generate Predictions

In [ ]:
artefact = joblib.load(os.path.join(MODELS_DIR, 'best_model.pkl'))
clf = artefact['clf']
le = artefact['le']

def get_features(df, split_name, model_tag='exp2_minilm_svm'):
    y_raw = df['global_misconception_index'].values
    mask = np.isin(y_raw, le.classes_)
    
    emb = np.load(os.path.join(MODELS_DIR, f'{model_tag}_emb_{split_name}.npy'))[mask]
    ts  = np.load(os.path.join(MODELS_DIR, f'ts_feats_{split_name}.npy'))[mask]
    
    from sklearn.preprocessing import normalize
    X = np.hstack([normalize(emb), ts])
    y = le.transform(y_raw[mask])
    return X, y, mask

X_test, y_test, mask_test = get_features(test_df, 'test')
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)
confidence = np.max(y_proba, axis=1)

# Pre-calculate top-3 for all
top3_preds = np.argsort(y_proba, axis=1)[:, -3:]

## 3. Training-Data Availability per Class

In [ ]:
stats = []
for cls in le.classes_:
    n_train = len(train_df[train_df['global_misconception_index'] == cls])
    n_val   = len(val_df[val_df['global_misconception_index'] == cls])
    n_test  = len(test_df[test_df['global_misconception_index'] == cls])
    probs   = all_df[all_df['global_misconception_index'] == cls]['problem_id'].nunique()
    
    stats.append({
        'misconception_id': cls,
        'total': n_train + n_val + n_test,
        'train': n_train,
        'val': n_val,
        'test': n_test,
        'unique_problems': probs
    })

stats_df = pd.DataFrame(stats)
print(f"Classes with <5 total examples: {len(stats_df[stats_df['total'] < 5])}")
print(f"Classes with <10 total examples: {len(stats_df[stats_df['total'] < 10])}")
print(f"Classes mapped to exactly 1 problem: {len(stats_df[stats_df['unique_problems'] == 1])}")

stats_df.sort_values('total', ascending=False).head()

## 4. Per-Class Metrics

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

class_metrics = []
for i, cls in enumerate(le.classes_):
    if str(i) not in report:
        continue
    
    rep = report[str(i)]
    mask_c = (y_test == i)
    if sum(mask_c) == 0: continue
        
    # Top-3 recall
    in_top3 = sum([1 for t3 in top3_preds[mask_c] if i in t3])
    top3_rec = in_top3 / sum(mask_c)
    
    class_metrics.append({
        'misconception_id': cls,
        'support': rep['support'],
        'precision': rep['precision'],
        'recall': rep['recall'],
        'f1': rep['f1-score'],
        'top3_recall': top3_rec
    })

metrics_df = pd.DataFrame(class_metrics).sort_values('f1')
display(metrics_df.head(10))
display(metrics_df.tail(10))

metrics_df.to_csv('../experiments/results/diagnosis/per_class_metrics.csv', index=False)

## 5. Confusion Analysis

In [ ]:
cm = confusion_matrix(y_test, y_pred)
confused = []
for i in range(len(le.classes_)):
    for j in range(i+1, len(le.classes_)):
        count_ij = cm[i, j]
        count_ji = cm[j, i]
        total = count_ij + count_ji
        if total > 0:
            cls_A = le.classes_[i]
            cls_B = le.classes_[j]
            confused.append({
                'class_A': cls_A,
                'class_B': cls_B,
                'A_to_B': count_ij,
                'B_to_A': count_ji,
                'total_confusion': total,
                'desc_A': misc_bank.get(cls_A, ""),
                'desc_B': misc_bank.get(cls_B, "")
            })

confused_df = pd.DataFrame(confused).sort_values('total_confusion', ascending=False)
display(confused_df.head(20))
confused_df.head(20).to_csv('../experiments/results/diagnosis/top_confused_pairs.csv', index=False)

## 6. Confidence Analysis

In [ ]:
conf_data = []
for idx in range(len(y_test)):
    true_lbl = le.inverse_transform([y_test[idx]])[0]
    pred_lbl = le.inverse_transform([y_pred[idx]])[0]
    conf = confidence[idx]
    correct = (y_test[idx] == y_pred[idx])
    in_top3 = (y_test[idx] in top3_preds[idx])
    
    conf_data.append({
        'true': true_lbl,
        'pred': pred_lbl,
        'confidence': conf,
        'correct': correct,
        'in_top3': in_top3
    })

conf_df = pd.DataFrame(conf_data)

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=conf_df, x='confidence', hue='correct', bins=20, multiple='stack', ax=ax)
plt.title('Confidence Distribution by Correctness')
plt.savefig('../experiments/results/diagnosis/confidence_dist.png', dpi=100)
plt.show()

print("High Confidence Incorrect (>0.8):", len(conf_df[(conf_df['confidence'] > 0.8) & (~conf_df['correct'])]))
print("Low Confidence Correct (<0.3):", len(conf_df[(conf_df['confidence'] < 0.3) & (conf_df['correct'])]))

## 7. Problem-Disjoint Comparison

In [ ]:
pd_clf_artefact = joblib.load(os.path.join(MODELS_DIR, 'exp5_fusion.pkl')) # we will compare with the main model or PD specific?
# Actually we trained a PD model in train_fusion: we should look at its results.
with open(os.path.join(RESULTS_DIR, 'exp5_fusion_test.json')) as f:
    res_normal = json.load(f)
with open(os.path.join(RESULTS_DIR, 'exp5_fusion_problem_disjoint.json')) as f:
    res_pd = json.load(f)

print("Normal Test Macro F1:", res_normal['macro_f1'])
print("Problem-Disjoint Macro F1:", res_pd['macro_f1'])

diff = {}
for cls, f1_norm in res_normal['per_class_f1'].items():
    if cls in res_pd['per_class_f1']:
        diff[cls] = f1_norm - res_pd['per_class_f1'][cls]

diff_df = pd.DataFrame(list(diff.items()), columns=['class', 'f1_drop']).sort_values('f1_drop', ascending=False)
print("Classes with highest performance drop in disjoint split:")
display(diff_df.head(10))